In [1]:
from pymongo import MongoClient
import os
from dotenv import load_dotenv

load_dotenv()

client = MongoClient(os.getenv('MONGODB_URI'))

# Lister toutes les bases
db_list = client.list_database_names()

for db_name in db_list:
    print(f"\n📌 Base : {db_name}")
    db = client[db_name]

    # Lister les collections de cette base
    collections = db.list_collection_names()

    if not collections:
        print("  (aucune collection)")
        continue

    for col in collections:
        count = db[col].count_documents({})
        print(f"  - {col} : {count} documents")

db = client['flight_delay_history_db']

#cursor = db['aviationstack_historical_landed_flights'].find().limit(100)
#df_aviation = pd.DataFrame(list(cursor))
#df_aviation.head()




📌 Base : flight_delay_db
  - aviationstack_flights : 1294 documents
  - afklm_flights : 1000 documents
  - weather_data : 70 documents

📌 Base : flight_delay_history_db
  - aviationstack_historical_landed_flights : 26 documents

📌 Base : flight_delays_test
  - test_collection : 22 documents

📌 Base : admin
  (aucune collection)

📌 Base : local
  - oplog.rs : 26 documents


In [ ]:
# Copier les bons vols
# A UTILISER UNE FOIS
# Source et destination
source_db = client['flight_delay_db']
target_db = client['flight_delay_history_db']

source_collection = source_db['aviationstack_flights']
target_collection = target_db['aviationstack_historical_landed_flights']

# Filtre : vols atterris avec arrival.actual non null
query = {
    "flight_status": "landed",
    "arrival.actual": {"$nin": [None, ""]}
}

# Récupération des documents filtrés
filtered_docs = list(source_collection.find(query))

# Insertion dans la collection cible
if filtered_docs:
    result = target_collection.insert_many(filtered_docs)
    print(f"{len(result.inserted_ids)} documents copiés.")
else:
    print("Aucun document à copier.")


24 documents copiés.


In [2]:
from pprint import pprint

db = client['flight_delay_history_db']

collection = db['aviationstack_historical_landed_flights']

# condition
query = {
    "flight_status": "landed",
    "arrival.actual": {"$ne": None}
}

# Compter les documents
count = collection.count_documents(query)
print("Nombre de documents filtrés :", count)

# Afficher les 1 derniers
for doc in collection.find(query).sort("collected_at", -1).limit(1):
    pprint(doc)


Nombre de documents filtrés : 26
{'_id': 'U24835_2026-02-01',
 'aircraft': {'iata': None,
              'icao': None,
              'icao24': '440037',
              'registration': None},
 'airline': {'iata': 'U2', 'icao': 'EZY', 'name': 'easyJet'},
 'arrival': {'actual': '2026-02-01T15:46:00+00:00',
             'actual_runway': '2026-02-01T15:46:00+00:00',
             'airport': 'Linate',
             'baggage': None,
             'delay': None,
             'estimated': '2026-02-01T15:54:00+00:00',
             'estimated_runway': '2026-02-01T15:46:00+00:00',
             'gate': None,
             'iata': 'LIN',
             'icao': 'LIML',
             'scheduled': '2026-02-01T15:50:00+00:00',
             'terminal': None,
             'timezone': 'Europe/Rome'},
 'collected_at': datetime.datetime(2026, 2, 1, 16, 38, 3, 895000),
 'departure': {'actual': '2026-02-01T14:40:00+00:00',
               'actual_runway': '2026-02-01T14:40:00+00:00',
               'airport': 'Orly',
  

In [4]:
from pymongo import MongoClient
import os
from dotenv import load_dotenv

load_dotenv()

client = MongoClient(os.getenv("MONGODB_URI"))
db = client["flight_delay_db"]
collection = db["aviationstack_flights"]

query = {
    "arrival.actual": {"$nin": [None, ""]}
}

count = collection.count_documents(query)
print("Nombre de documents :", count)


Nombre de documents : 24


In [ ]:
# test ML pas à pas
import os
from dotenv import load_dotenv
from pymongo import MongoClient
import pandas as pd

# ETAPE 0  : RECUPERATION DES DONNEES
load_dotenv()

client = MongoClient(os.getenv("MONGODB_URI"))
db = client["flight_delay_history_db"]
collection = db["aviationstack_historical_landed_flights"]

# Filtre : vols atterris avec données complètes minimales
query = {
    "flight_status": "landed",
    "arrival.actual": {"$nin": [None, ""]},
    "arrival.scheduled": {"$nin": [None, ""]},
    "departure.scheduled": {"$nin": [None, ""]},
    "departure.actual": {"$nin": [None, ""]},
}

docs = list(collection.find(query))



# ETAPE 1  : APPLATISSEMENT DES DONNEES
# 🔹 Aplatir les documents imbriqués
df_flat = pd.json_normalize(
    docs,
    sep="_"  # airline.name -> airline_name
)

# 🔹 Garder / renommer les colonnes utiles
cols_to_keep = [
    "_id",
    "flight_date",
    "flight_status",
    "collected_at",
    "filtered_at",
    "live",

    # Airline
    "airline_name",
    "airline_iata",
    "airline_icao",

    # Aircraft
    "aircraft_registration",
    "aircraft_iata",
    "aircraft_icao",
    "aircraft_icao24",

    # Departure
    "departure_airport",
    "departure_timezone",
    "departure_iata",
    "departure_icao",
    "departure_terminal",
    "departure_gate",
    "departure_scheduled",
    "departure_estimated",
    "departure_actual",
    "departure_estimated_runway",
    "departure_actual_runway",

    # Arrival
    "arrival_airport",
    "arrival_timezone",
    "arrival_iata",
    "arrival_icao",
    "arrival_terminal",
    "arrival_gate",
    "arrival_baggage",
    "arrival_scheduled",
    "arrival_estimated",
    "arrival_actual",
    "arrival_estimated_runway",
    "arrival_actual_runway",

    # Flight
    "flight_number",
    "flight_iata",
    "flight_icao",
    "flight_codeshared",
]

# Certaines colonnes peuvent ne pas exister pour tous les docs → on intersecte
existing_cols = [c for c in cols_to_keep if c in df_flat.columns]
df_flat = df_flat[existing_cols]

# Optionnel : conversion de quelques champs en datetime
datetime_cols = [
    "flight_date",
    "collected_at",
    "filtered_at",
    "departure_scheduled",
    "departure_estimated",
    "departure_actual",
    "departure_estimated_runway",
    "departure_actual_runway",
    "arrival_scheduled",
    "arrival_estimated",
    "arrival_actual",
    "arrival_estimated_runway",
    "arrival_actual_runway",
]

for col in datetime_cols:
    if col in df_flat.columns:
        df_flat[col] = pd.to_datetime(df_flat[col], errors="coerce")

# df_flat.head()

# ETAPE 2  : ANALYSE DES VALEURS NULLS
# Nombre de valeurs nulles par colonne
null_counts = df_flat.isnull().sum()

# Pourcentage de valeurs nulles par colonne
null_percent = (null_counts / len(df_flat)) * 100

# Résumé filtré : uniquement les colonnes avec plus de 30% de valeurs nulles
null_summary = (
    pd.DataFrame({
        "null_count": null_counts,
        "null_percent": null_percent.round(2)
    })
    .query("null_percent > 30")
    .sort_values(by="null_percent", ascending=False)
)

display(null_summary)

# On supprimme les colonnes qui ont plus de 30% de valeurs nulls
# Colonnes à supprimer : null_percent > 30%
cols_to_drop = null_summary[null_summary["null_percent"] > 30.0].index.tolist()

# Nouveau DataFrame nettoyé
df_flat_filtered = df_flat.drop(columns=cols_to_drop)

print(f"Colonnes supprimées : {len(cols_to_drop)}")
print(cols_to_drop)

df_flat_filtered.head(1)

AttributeError: 'DataFrame' object has no attribute 'limit'

In [ ]:
# test ML
import os
from dotenv import load_dotenv
from pymongo import MongoClient
import pandas as pd
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

load_dotenv()

client = MongoClient(os.getenv("MONGODB_URI"))
db = client["flight_delay_history_db"]
collection = db["aviationstack_historical_landed_flights"]

# Filtre : vols atterris avec données complètes
query = {
    "flight_status": "landed",
    "arrival.actual": {"$nin": [None, ""]},
    "arrival.scheduled": {"$nin": [None, ""]},
    "departure.scheduled": {"$nin": [None, ""]},
    "departure.actual": {"$nin": [None, ""]},
}

docs = list(collection.find(query))
df = pd.DataFrame(docs)
#df.head()
# -----------------------------
# Feature engineering
# -----------------------------

def to_dt(x):
    try:
        return pd.to_datetime(x)
    except:
        return None

# Délais départ
df["dep_scheduled"] = df["departure"].apply(lambda x: to_dt(x.get("scheduled")))
df["dep_actual"] = df["departure"].apply(lambda x: to_dt(x.get("actual")))
df["dep_estimated"] = df["departure"].apply(lambda x: to_dt(x.get("estimated")))

df["departure_delay_actual"] = (df["dep_actual"] - df["dep_scheduled"]).dt.total_seconds() / 60
df["departure_delay_estimated"] = (df["dep_estimated"] - df["dep_scheduled"]).dt.total_seconds() / 60
df["departure_delay_actual"] = df["departure_delay_actual"].apply(lambda x: max(x, 0))
df["departure_delay_estimated"] = df["departure_delay_estimated"].apply(lambda x: max(x, 0))


# Délais arrivée
df["arr_scheduled"] = df["arrival"].apply(lambda x: to_dt(x.get("scheduled")))
df["arr_actual"] = df["arrival"].apply(lambda x: to_dt(x.get("actual")))
df["arr_estimated"] = df["arrival"].apply(lambda x: to_dt(x.get("estimated")))

df["arrival_delay_actual"] = (df["arr_actual"] - df["arr_scheduled"]).dt.total_seconds() / 60
df["arrival_delay_actual"] = df["arrival_delay_actual"].apply(lambda x: max(x, 0))

df["arrival_delay_estimated"] = (df["arr_estimated"] - df["arr_scheduled"]).dt.total_seconds() / 60
df["arrival_delay_estimated"] = df["arrival_delay_estimated"].apply(lambda x: max(x, 0))

# Cible
df = df.dropna(subset=["arrival_delay_actual"])
y = df["arrival_delay_actual"]

# Features temporelles
df["flight_date"] = pd.to_datetime(df["flight_date"])
df["day_of_week"] = df["flight_date"].dt.weekday
df["month"] = df["flight_date"].dt.month
df["hour_dep"] = df["dep_scheduled"].dt.hour

# Features catégorielles
df["airline_iata"] = df["airline"].apply(lambda x: x.get("iata"))
df["dep_iata"] = df["departure"].apply(lambda x: x.get("iata"))
df["arr_iata"] = df["arrival"].apply(lambda x: x.get("iata"))
df["aircraft_icao24"] = df["aircraft"].apply(lambda x: x.get("icao24"))
df["flight_number"] = df["flight"].apply(lambda x: x.get("number"))

df.head()
# Dataset final
features = df[
    [
        "departure_delay_actual",
        "departure_delay_estimated",
        "arrival_delay_estimated",
        "day_of_week",
        "month",
        "hour_dep",
        "airline_iata",
        "dep_iata",
        "arr_iata",
        "aircraft_icao24",
        "flight_number",
    ]
]

# Encodage
X = pd.get_dummies(features)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Modèle
model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

# Prédictions
y_pred = model.predict(X_test)

# Évaluation
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("📊 Évaluation du modèle :")
print(f"MAE  : {mae:.2f} minutes")
print(f"RMSE : {rmse:.2f} minutes")
print(f"R²   : {r2:.2f}")


,_id,aircraft,airline,arrival,collected_at,departure,filtered_at,flight,flight_date,flight_status,...,arrival_delay_actual,arrival_delay_estimated,day_of_week,month,hour_dep,airline_iata,dep_iata,arr_iata,aircraft_icao24,flight_number
0,VY8015_2026-02-01,"{'registration': None, 'iata': None, 'icao': N...","{'name': 'Vueling', 'iata': 'VY', 'icao': 'VLG'}","{'airport': 'El Prat De Llobregat', 'timezone'...",2026-02-01 16:38:03.737,"{'airport': 'Orly', 'timezone': 'Europe/Paris'...",2026-02-01 16:34:45.588,"{'number': '8015', 'iata': 'VY8015', 'icao': '...",2026-02-01,landed,...,0.0,0.0,6,2,15,VY,ORY,BCN,345043,8015
1,U24835_2026-02-01,"{'registration': None, 'iata': None, 'icao': N...","{'name': 'easyJet', 'iata': 'U2', 'icao': 'EZY'}","{'airport': 'Linate', 'timezone': 'Europe/Rome...",2026-02-01 16:38:03.895,"{'airport': 'Orly', 'timezone': 'Europe/Paris'...",2026-02-01 16:34:45.626,"{'number': '4835', 'iata': 'U24835', 'icao': '...",2026-02-01,landed,...,0.0,4.0,6,2,14,U2,ORY,LIN,440037,4835
2,AP1026_2025-12-22,"{'registration': None, 'iata': None, 'icao': N...","{'name': 'AlbaStar', 'iata': 'AP', 'icao': 'LAV'}","{'airport': 'Barajas', 'timezone': 'Europe/Mad...",2025-12-22 22:54:02.401,"{'airport': 'Orly', 'timezone': 'Europe/Paris'...",NaT,"{'number': '1026', 'iata': 'AP1026', 'icao': '...",2025-12-22,landed,...,0.0,0.0,0,12,19,AP,ORY,MAD,346107,1026
3,QR3758_2025-12-22,"{'registration': None, 'iata': None, 'icao': N...","{'name': 'Qatar Airways', 'iata': 'QR', 'icao'...","{'airport': 'El Prat De Llobregat', 'timezone'...",2025-12-22 22:54:02.442,"{'airport': 'Orly', 'timezone': 'Europe/Paris'...",NaT,"{'number': '3758', 'iata': 'QR3758', 'icao': '...",2025-12-22,landed,...,0.0,0.0,0,12,21,QR,ORY,BCN,342592,3758
4,LL5629_2025-12-22,"{'registration': None, 'iata': None, 'icao': N...","{'name': 'Miami Air International', 'iata': 'L...","{'airport': 'El Prat De Llobregat', 'timezone'...",2025-12-22 22:54:02.463,"{'airport': 'Orly', 'timezone': 'Europe/Paris'...",NaT,"{'number': '5629', 'iata': 'LL5629', 'icao': '...",2025-12-22,landed,...,0.0,0.0,0,12,21,LL,ORY,BCN,342592,5629
